# Implement K-Nearest Neighbors (KNN) and evaluate model performance

## 1. Import Required Libraries
Import necessary libraries such as pandas, numpy, and scikit-learn modules for models and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Load and Prepare Data
Load the dataset `data.xls`, handle missing values, encode categoricals, split features and target, and apply feature scaling. Feature scaling is essential for distance-based algorithms like KNN.

In [ ]:
# Load the dataset
# Note: Ensure you have xlrd installed to read older .xls files. 
df = pd.read_excel('data.xls')

# Basic Preprocessing
# Drop any missing values for simplicity
df.dropna(inplace=True)

# Encode categorical variables
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# Assuming the last column is the target variable
# If not, you can replace 'df.columns[-1]' with the actual target column name
X = df.drop(df.columns[-1], axis=1)
y = df.iloc[:, -1]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling (Crucial for KNN!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

df.head()

## 3. Implement K-Nearest Neighbors Classifier
Initialize and train a KNeighborsClassifier with an initial `n_neighbors` (k) value on the scaled training data. generating predictions on the test set.

In [ ]:
# Initialize the KNN classifier with an arbitrary value like k=5
knn_classifier = KNeighborsClassifier(n_neighbors=5)
knn_classifier.fit(X_train_scaled, y_train)

# Make predictions
knn_predictions = knn_classifier.predict(X_test_scaled)

## 4. Evaluate KNN Model
Calculate and display evaluation metrics including accuracy score, confusion matrix, and classification report for the default `k=5` KNN Model.

In [ ]:
knn_accuracy = accuracy_score(y_test, knn_predictions)
print(f"KNN Accuracy (k=5): {knn_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_test, knn_predictions))

plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, knn_predictions), annot=True, fmt='d', cmap='Oranges')
plt.title('KNN Confusion Matrix (k=5)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 5. Finding the Optimal K (Elbow Method)
Loop through various values of K and calculate the error rate to find the best parameter that minimizes error without overfitting.

In [ ]:
error_rate = []

# Testing K values from 1 to 40
for i in range(1, 40):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_scaled, y_train)
    pred_i = knn.predict(X_test_scaled)
    error_rate.append(np.mean(pred_i != y_test))

plt.figure(figsize=(10,6))
plt.plot(range(1, 40), error_rate, color='blue', linestyle='dashed', marker='o',
         markerfacecolor='red', markersize=8)
plt.title('Error Rate vs. K Value')
plt.xlabel('K')
plt.ylabel('Error Rate')
plt.show()

# Find the best based on min error
best_k = error_rate.index(min(error_rate)) + 1
print(f"Optimal number of neighbors based on minimum error rate is approximately K = {best_k}")

## 6. Evaluate Model with Optimal K
Re-train the KNN model using the optimal $K$ found from the Elbow method.

In [ ]:
# Retrain with the optimal K value Let's say it's best_k
knn_optimal = KNeighborsClassifier(n_neighbors=best_k)
knn_optimal.fit(X_train_scaled, y_train)
pred_optimal = knn_optimal.predict(X_test_scaled)

optimal_accuracy = accuracy_score(y_test, pred_optimal)

print(f"KNN Accuracy (k={best_k}): {optimal_accuracy:.4f}")
print('\nClassification Report:')
print(classification_report(y_test, pred_optimal))

plt.figure(figsize=(5,4))
sns.heatmap(confusion_matrix(y_test, pred_optimal), annot=True, fmt='d', cmap='Oranges')
plt.title(f'KNN Confusion Matrix (k={best_k})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()